# Agent Bricks by code: Supervisor Agent with human-in-the-loop

**Everything by code, on serverless compute.** This notebook:
1. Creates a **Unity Catalog function** as a governed action tool (`issue_refund`)
2. Creates a **Supervisor Agent** coordinating 2 sub-agents/tools: the `customer genie` Genie Space and the UC function
3. Tests it end-to-end with **human-in-the-loop** approval before any refund is executed
4. **Autologs every call as MLflow traces**
5. Cleans everything up at the end

**Prerequisites**
- Serverless compute attached to this notebook
- Preview enabled: *Mosaic AI Agent Bricks*
- An existing Genie Space (`customer genie`) over the `demo.demo` tables


In [0]:
%pip install databricks-sdk==0.120.0 databricks-langchain==0.20.0 langchain==1.3.13 langchain-openai==1.3.5 mlflow==3.14.0
%restart_python

In [0]:
%run ../_config/config_unity_catalog

In [0]:
%run ../_config/config_multiagent

In [0]:
CATALOG = catalog
SCHEMA = schema

# ID of your existing Genie Space (Genie UI > your space > URL: /genie/rooms/<this-id>)
GENIE_SPACE_ID = "01f129b87e20108fad4b9f084cfb50ac"

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("Workspace:", w.config.host)

## Step 1 — Action tool: a Unity Catalog function
`issue_refund` is the *sensitive action* of this demo: the supervisor may only
call it **after explicit human approval**. Here it returns a confirmation ID;
in production it would write to a Delta table or call a payment API.
Note the function and parameter COMMENTs: agents read them to understand the
tool, exactly like Genie reads column comments.


In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.issue_refund(
    order_id INT COMMENT 'The order to refund',
    amount DOUBLE COMMENT 'Refund amount in USD',
    reason STRING COMMENT 'Business justification for the refund'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Issues a customer refund and returns the confirmation ID. SENSITIVE ACTION: only call after explicit human approval.'
AS $$
    import hashlib
    h = hashlib.sha1(f"{{order_id}}-{{amount}}-{{reason}}".encode()).hexdigest()[:8].upper()
    return f"REFUND-{{h}} registered: order {{order_id}}, amount ${{amount:.2f}} ({{reason}})"
$$
""")
print(f"UC function {CATALOG}.{SCHEMA}.issue_refund created")

# Sanity check
display(spark.sql(f"SELECT {CATALOG}.{SCHEMA}.issue_refund(1001, 399.99, 'defective monitor')"))

## Step 2 — Create the Supervisor Agent (SDK)
One supervisor, two tools. Descriptions drive delegation; instructions
enforce the human-in-the-loop policy: *propose first, execute only after an
explicit approval*.


In [0]:
from databricks.sdk.service.supervisoragents import (
    SupervisorAgent,
    Tool,
    GenieSpace,
    UcFunction
)

sup = w.supervisor_agents.create_supervisor_agent(
    supervisor_agent=SupervisorAgent(
        display_name="customer supervisor",
        description=(
            "Answers customer questions by combining order data (Genie), "
            "TechStore policies (Knowledge Assistant), and can issue refunds "
            "after human approval."
        ),
        instructions=(
            "You coordinate customer operations. "
            "Use the genie space for any question about customers, orders, "
            "order items, products or shipping zones. "
            "HUMAN-IN-THE-LOOP POLICY: issue_refund is a sensitive action. "
            "Never call it directly. First present the exact order id, amount "
            "and reason, then ask for explicit approval. Only call issue_refund "
            "after the user replies with an explicit approval such as "
            "'I approve'. Never disclose customer registration dates."
        ),
    )
)
print("Supervisor created:", sup.name)
print("Endpoint:", sup.endpoint_name)
print("MLflow experiment:", sup.experiment_id)

In [0]:
tools = [
    (
        "customer_genie",
        Tool(
            tool_type="genie_space",
            genie_space=GenieSpace(id=GENIE_SPACE_ID),
            description=(
                "Answers analytical questions about customers, orders, order "
                "items, products and shipping zones by generating SQL on the "
                "demo.demo tables."
            ),
        ),
    ),
    (
        "issue_refund",
        Tool(
            tool_type="uc_function",
            uc_function=UcFunction(name=f"{CATALOG}.{SCHEMA}.issue_refund"),
            description=(
                "Issues a customer refund and returns a confirmation ID. "
                "SENSITIVE: only invoke after the user gave explicit approval "
                "of the exact amount."
            ),
        ),
    ),
]


for tool_id, tool in tools:
    created = w.supervisor_agents.create_tool(parent=sup.name, tool=tool, tool_id=tool_id)
    print("Tool attached:", created.name)

In [0]:
import time

while True:
    ep = w.serving_endpoints.get(sup.endpoint_name)
    state = ep.state.ready.value if ep.state and ep.state.ready else "UNKNOWN"
    print("Endpoint state:", state)
    if state == "READY":
        break
    time.sleep(30)

## Step 3 — MLflow autolog (LangChain)
`mlflow.langchain.autolog()` traces every LangChain invocation: inputs,
outputs, latency, and the delegation visible in the **Traces** tab of the
experiment. The supervisor also logs its own traces in its Agent Bricks
experiment (`sup.experiment_id`).

Agent Bricks endpoints implement the **Responses API** (`input` field, not
`messages`), so we use `ChatOpenAI` with `use_responses_api=True`, pointed at
the workspace serving endpoints.


In [0]:
import mlflow

mlflow.langchain.autolog()

# Optional: send traces to a dedicated experiment instead of the notebook's
# mlflow.set_experiment("/Shared/agent-bricks-code-demo")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

# Agent Bricks endpoints implement the Responses API ('input' field, not
# 'messages'), so we use ChatOpenAI with use_responses_api=True, pointed at
# the workspace serving endpoints.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

llm = ChatOpenAI(
    model=sup.endpoint_name,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
    use_responses_api=True,
)

## Step 4 — Test 1: refund request → Genie lookup + proposal
"John Smith wants a refund for his last order." The supervisor must find the
order (Genie) and then *propose* the refund — not execute it.


In [0]:
def ask(messages):
    """Query the supervisor endpoint through LangChain and return the reply text."""
    resp = llm.invoke(messages)
    # In Responses API mode, content may be a list of content blocks
    if isinstance(resp.content, str):
        return resp.content
    return "".join(
        b.get("text", "") for b in resp.content if isinstance(b, dict)
    )

history = [
    HumanMessage(
        "John Smith wants a refund for his last order. Can you help me ?"
    )
]

answer = ask(history)
history.append(AIMessage(answer))
print(answer)


Read the answer above: the `<name>customer_genie</name>` block shows the
delegation — Genie generated the SQL and returned order **2001, $150.00**.
The supervisor then built a refund summary and **stopped for approval**:
`issue_refund` was NOT called (check the trace).

> Note: the endpoint is stateless — we re-send the full `history` each turn,
> and the supervisor may re-query Genie to re-ground itself. The full
> sequence is visible in each trace.


## Step 5 — Test 2: add the business reason
**Turn 2** — we provide the reason (defective monitor). Still a proposal:
order ID, amount, reason, and an explicit approval request.


In [0]:
history.append(
    HumanMessage("The monitor is defective (dead pixels). Please refund this order.")
)

proposal = ask(history)
history.append(AIMessage(proposal))
print(proposal)

☝️ **The human checkpoint is now.** Read the proposal above: order ID,
amount, reason. Nothing has been executed — `issue_refund` was not called
(check the trace). Run the next cell **only if you approve**.


In [0]:
history.append(HumanMessage("I approve, issue the refund."))

confirmation = ask(history)
history.append(AIMessage(confirmation))
print(confirmation)

**Turn 3** executed the action: the trace shows the `issue_refund` UC
function call and the confirmation ID (`REFUND-...`). Two layers of control:
the instructions force a proposal first, and the action only runs in a turn
where a human explicitly approved.

> **Note — platform-native HITL:** with the Supervisor API in background
> mode, MCP tool calls *require* an `mcp_approval_request` /
> `mcp_approval_response` exchange enforced by Databricks itself.


## Step 6 — Review the traces in MLflow


In [0]:
print("Notebook traces:   MLflow UI > this notebook's experiment > Traces tab")
print(f"Supervisor traces: experiment_id = {sup.experiment_id}")
print(f"Agents UI:         {w.config.host}/agents")

# Programmatic access to the traces:
traces = mlflow.search_traces(max_results=5)
display(traces)

## Step 7 — Cleanup
Deleting the supervisor removes its tools and its serving endpoint. Set
`CLEANUP = True` to leave the workspace as clean as you found it.


In [0]:
CLEANUP = False  # set to True to delete everything created by this notebook

if CLEANUP:
    # 1. Supervisor Agent (tools are deleted with it, endpoint is removed)
    w.supervisor_agents.delete_supervisor_agent(name=sup.name)
    print("Deleted supervisor:", sup.name)

    # 2. UC function
    spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.issue_refund")
    print(f"Dropped function {CATALOG}.{SCHEMA}.issue_refund")

    print("Cleanup complete.")
else:
    print("CLEANUP is False - nothing deleted.")